# Jev API + BipedalWalker — v4 gait-preserving supervisor

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/simple-jev/blob/main/notebooks/Jev_API_BipedalWalker_Controller_Colab.ipynb)

Jev selects a bounded gait-speed setting; Gymnasium's stateful heuristic generates joint actions at every physics step. This is a hybrid supervisor, not a learned or generic Jev motor policy.

v3 blended a stepping action with a symmetric standing action and rate-limited joint commands. Those interventions interrupted foot placement and caused early falls. v4 preserves the full stepping action, uses a 100-frame startup, and keeps all four Jev speed choices close to the tested 0.26 setting. The episode limit is 2,000 frames and genuine terminal states still stop the run.

Store your TypeSafe key in Colab Secrets as `TYPESAFE_API_KEY`. Offline tests below do not call the API. Live Jev performance depends on its decisions; offline tests are not evidence of live API performance.


In [ ]:
#@title 1. Install dependencies
!pip -q install "gymnasium[box2d]==1.3.0" imageio imageio-ffmpeg typesafe-sdk pillow matplotlib


In [ ]:
#@title 2. Imports, API key, and configuration
import os, time, json, getpass
from collections import deque
from pathlib import Path

import numpy as np
import gymnasium as gym
import imageio.v2 as imageio
import matplotlib.pyplot as plt

from PIL import Image as PILImage
from IPython.display import Video, Image, display
from gymnasium.envs.box2d.bipedal_walker import BipedalWalkerHeuristics
from typesafe_sdk import Choice, TypeSafeClient

def load_typesafe_key():
    key = None
    try:
        from google.colab import userdata
        key = userdata.get("TYPESAFE_API_KEY")
    except Exception:
        pass
    if not key:
        key = os.environ.get("TYPESAFE_API_KEY")
    if not key:
        key = getpass.getpass("TypeSafe API key from https://typesafe.ai: ").strip()
    if not key:
        raise RuntimeError("No TYPESAFE_API_KEY supplied.")
    return key

os.environ["TYPESAFE_API_KEY"] = load_typesafe_key()
os.environ.setdefault("TYPESAFE_DEFAULT_MODEL", "jev-latest")
client = TypeSafeClient()

SEED = 0
MAX_STEPS = 2000
DECISION_EVERY = 5
MIN_CONFIDENCE = 0.40

TARGET_SPEED = 0.26
STARTUP_STEPS = 100
MODE_SPEED = {"cruise": 0.260, "cautious": 0.258, "balance": 0.256, "recover": 0.254}
SPEED_SOFT = 0.36
SPEED_HARD = 0.48

VIDEO_EVERY = 4
GIF_EVERY = 8
OUTPUT_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
VIDEO_PATH = str(OUTPUT_DIR / "jev_bipedalwalker_v4.mp4")
GIF_PATH = str(OUTPUT_DIR / "jev_bipedalwalker_v4.gif")
LOG_PATH = str(OUTPUT_DIR / "jev_bipedalwalker_v4_decisions.json")

print("✓ TypeSafe key loaded")
print("✓ model:", os.environ["TYPESAFE_DEFAULT_MODEL"])
print("✓ Jev interval:", DECISION_EVERY, "frames; motor loop: 50 Hz")

In [ ]:
#@title 3. Jev supervisor + gait-preserving controller
MODE_CRITERIA = {
    "cruise": "Regular stepping at speed parameter 0.260 when stable.",
    "cautious": "Slightly lower speed parameter 0.258 when uncertain.",
    "balance": "Preserve stepping at 0.256 when tilt or oscillation is growing.",
    "recover": "Preserve foot placement at 0.254 when instability is large. This cannot guarantee recovery.",
}

def jev_choose_mode(state):
    t0 = time.perf_counter()
    response = client.system_one(
        state=state,
        questions={
            "control_mode": Choice(
                instructions=(
                    "Choose one supervisory mode for Gymnasium BipedalWalker-v3. "
                    "React early to predicted angle, angular velocity, excessive "
                    "forward speed, and recent trend. Priorities: avoid hull-ground "
                    "contact, stay upright, then walk right smoothly."
                ),
                criteria=MODE_CRITERIA,
            )
        },
    )
    latency_ms = (time.perf_counter() - t0) * 1000.0
    a = response.answers["control_mode"]
    return {
        "mode": a.choice,
        "confidence": float(a.confidence),
        "probabilities": dict(a.probabilities),
        "latency_ms": latency_ms,
        "model": response.model,
    }

def risk_metrics(s):
    angle = float(s[0])
    omega = float(s[1])
    vx = float(s[2])
    vy = float(s[3])

    # obs[1] = 2 * angular_velocity / 50, so 100 ms projection adds 2.5*obs[1].
    pred = angle + 2.5 * omega

    forward = float(np.clip((-pred - 0.06) / 0.24, 0.0, 1.0))
    backward = float(np.clip((pred - 0.10) / 0.24, 0.0, 1.0))
    tilt = max(forward, backward)
    speed = float(np.clip((vx - TARGET_SPEED) / (SPEED_HARD - TARGET_SPEED), 0.0, 1.0))
    vertical = float(np.clip((abs(vy) - 0.10) / 0.30, 0.0, 1.0))
    risk = float(max(tilt, 0.75 * speed, 0.55 * vertical))

    return {
        "angle": angle, "omega": omega, "vx": vx, "vy": vy,
        "pred": float(pred), "tilt_risk": tilt,
        "speed_risk": speed, "vertical_risk": vertical, "risk": risk,
    }

def new_reference():
    reference = BipedalWalkerHeuristics()
    # Explicit instance state: do not share the class-level action array.
    reference.state = reference.STAY_ON_ONE_LEG
    reference.moving_leg = 0
    reference.supporting_leg = 1
    reference.supporting_knee_angle = reference.SUPPORT_KNEE_ANGLE
    reference.a = np.zeros(4, dtype=np.float32)
    reference.SPEED = TARGET_SPEED
    return reference


def gait_action(reference, obs, step, requested_mode):
    mode = requested_mode if requested_mode in MODE_SPEED else "cautious"
    reason = "jev" if step >= STARTUP_STEPS else "startup"
    reference.SPEED = MODE_SPEED[mode] if step >= STARTUP_STEPS else TARGET_SPEED
    # No standing blend, torque attenuation, or slew limiter: preserve foot placement.
    action = np.asarray(reference.step_heuristic(obs), dtype=np.float32).copy()
    if action.shape != (4,) or not np.isfinite(action).all():
        raise RuntimeError("Invalid gait action; stop rather than send invalid torques.")
    return np.clip(action, -1.0, 1.0), mode, reason, risk_metrics(obs)

def state_for_jev(obs, step, total_reward, recent_rewards, previous_mode):
    m = risk_metrics(obs)
    rr = list(recent_rewards)
    return {
        "environment": "Gymnasium BipedalWalker-v3",
        "goal": "walk right smoothly while preventing hull-ground contact",
        "step": int(step),
        "previous_mode": previous_mode,
        "mode_speed_parameters": MODE_SPEED,
        "startup_active": step < STARTUP_STEPS,
        "control_contract": "Modes only adjust gait speed; all preserve stepping. Risk is a diagnostic, not a calibrated fall probability.",
        "total_reward": round(float(total_reward), 3),
        "recent_reward_mean": round(float(np.mean(rr)) if rr else 0.0, 4),
        "hull": {
            "angle": round(m["angle"], 4),
            "angular_velocity_scaled": round(m["omega"], 4),
            "predicted_angle_100ms": round(m["pred"], 4),
            "horizontal_speed_scaled": round(m["vx"], 4),
            "vertical_speed_scaled": round(m["vy"], 4),
        },
        "risk": {
            "overall": round(m["risk"], 3),
            "tilt": round(m["tilt_risk"], 3),
            "speed": round(m["speed_risk"], 3),
            "vertical": round(m["vertical_risk"], 3),
            "target_speed": TARGET_SPEED,
            "soft_speed_limit": SPEED_SOFT,
            "hard_speed_limit": SPEED_HARD,
        },
        "contacts": {"leg0": bool(obs[8] > 0.5), "leg1": bool(obs[13] > 0.5)},
    }

test = jev_choose_mode({
    "environment": "connectivity test",
    "hull": {"angle": 0.0, "predicted_angle_100ms": 0.0, "horizontal_speed_scaled": 0.1},
    "risk": {"overall": 0.0},
})
print("✓ Jev API:", test["mode"], "confidence", round(test["confidence"], 3))

In [ ]:
#@title 4. Run v4 gait-preserving controller
env = gym.make("BipedalWalker-v3", render_mode="rgb_array", max_episode_steps=MAX_STEPS)
obs, info = env.reset(seed=SEED)

reference = new_reference()

requested_mode = "cruise"
recent_rewards = deque(maxlen=30)

total_reward = 0.0
decision_log, frame_log = [], []
reward_history, speed_history, angle_history = [], [], []
predicted_history, risk_history = [], []

api_failures = 0
low_confidence_fallbacks = 0
startup_frames = 0
terminated = truncated = False
start_x = float(env.unwrapped.hull.position.x)
end_x = start_x
fallen = False
gif_frames = []

writer = imageio.get_writer(
    VIDEO_PATH,
    format="FFMPEG",
    mode="I",
    fps=50.0 / VIDEO_EVERY,
    codec="libx264",
    pixelformat="yuv420p",
    macro_block_size=1,
)

try:
    for step in range(MAX_STEPS):
        if step % DECISION_EVERY == 0:
            state = state_for_jev(obs, step, total_reward, recent_rewards, requested_mode)
            try:
                d = jev_choose_mode(state)
                mode = d["mode"]
                if mode not in MODE_CRITERIA or not np.isfinite(d["confidence"]) or not 0 <= d["confidence"] <= 1:
                    raise RuntimeError(f"Unknown Jev mode: {mode}")
                if d["confidence"] < MIN_CONFIDENCE:
                    mode = "cautious"
                    low_confidence_fallbacks += 1
                requested_mode = mode
                m = risk_metrics(obs)

                decision_log.append({
                    "step": step, "raw_jev_mode": d["mode"], "jev_mode": requested_mode,
                    "confidence": d["confidence"],
                    "probabilities": d["probabilities"],
                    "latency_ms": d["latency_ms"],
                    **m, "total_reward": float(total_reward),
                })

                if len(decision_log) <= 10 or len(decision_log) % 10 == 0:
                    print(
                        f"decision {len(decision_log):03d} | step {step:04d} | "
                        f"Jev={requested_mode:>8s} | conf {d['confidence']:.3f} | "
                        f"{d['latency_ms']:.0f} ms | vx {m['vx']:+.3f} | "
                        f"angle {m['angle']:+.3f} | pred {m['pred']:+.3f} | risk {m['risk']:.2f}"
                    )
            except Exception as e:
                api_failures += 1
                requested_mode = "cautious"
                print("Jev API error -> cautious:", repr(e))

        action, applied_mode, reason, metrics = gait_action(reference, obs, step, requested_mode)
        startup_frames += int(reason == "startup")

        frame_log.append({
            "step": step, "jev_mode": requested_mode,
            "applied_mode": applied_mode, "override_reason": reason,
            "gait_speed": float(reference.SPEED), **metrics,
            "action": [float(x) for x in action],
        })

        obs, reward, terminated, truncated, info = env.step(action)
        end_x = float(env.unwrapped.hull.position.x)
        fallen = bool(env.unwrapped.game_over or end_x < 0)
        total_reward += float(reward)
        recent_rewards.append(float(reward))

        ma = risk_metrics(obs)
        reward_history.append(total_reward)
        speed_history.append(ma["vx"])
        angle_history.append(ma["angle"])
        predicted_history.append(ma["pred"])
        risk_history.append(ma["risk"])

        if step % VIDEO_EVERY == 0:
            frame = env.render()
            writer.append_data(frame)

        if step % GIF_EVERY == 0:
            frame = env.render()
            gif_frames.append(np.asarray(PILImage.fromarray(frame).resize((450, 300))))

        if terminated or truncated:
            print(f"Episode ended at step {step + 1} | terminated={terminated} truncated={truncated}")
            break
finally:
    writer.close()
    env.close()

if gif_frames:
    imageio.mimsave(GIF_PATH, gif_frames, duration=1000.0 * GIF_EVERY / 50.0, loop=0)

Path(LOG_PATH).write_text(
    json.dumps({"summary": {"steps": len(reward_history), "reward": total_reward, "distance": end_x-start_x, "terminated": bool(terminated), "truncated": bool(truncated), "seed": SEED}, "decisions": decision_log, "frames": frame_log}, indent=2),
    encoding="utf-8",
)

print("\n=== RESULT ===")
print("steps:", len(reward_history))
print("total reward:", round(total_reward, 2))
print("Jev decisions:", len(decision_log))
print("low-confidence fallbacks:", low_confidence_fallbacks)
print("API failures:", api_failures)
print("startup frames:", startup_frames)
print("distance (world units):", round(end_x - start_x, 2))
print("stop reason:", "fall" if fallen else "course finished" if terminated else "time limit" if truncated else "configured frame cap")

if decision_log:
    lat = [d["latency_ms"] for d in decision_log]
    print("mean Jev latency (ms):", round(float(np.mean(lat)), 1))
    print("p95 Jev latency (ms):", round(float(np.percentile(lat, 95)), 1))

In [ ]:
#@title 5. Video, GIF fallback, and diagnostics
print("MP4:")
display(Video(VIDEO_PATH, embed=True, html_attributes="controls loop"))

print("GIF fallback:")
display(Image(filename=GIF_PATH))

plt.figure(figsize=(12, 4))
plt.plot(reward_history)
plt.xlabel("step"); plt.ylabel("cumulative reward")
plt.title("v4 cumulative reward"); plt.grid(True, alpha=0.25); plt.show()

plt.figure(figsize=(12, 4))
plt.plot(speed_history, label="horizontal speed")
plt.axhline(TARGET_SPEED, linestyle="--", linewidth=1, label="target")
plt.axhline(SPEED_SOFT, linestyle="--", linewidth=1, label="soft limit")
plt.axhline(SPEED_HARD, linestyle="--", linewidth=1, label="hard limit")
plt.xlabel("step"); plt.ylabel("scaled speed")
plt.title("Observed speed (diagnostic only)"); plt.legend(); plt.grid(True, alpha=0.25); plt.show()

plt.figure(figsize=(12, 4))
plt.plot(angle_history, label="hull angle")
plt.plot(predicted_history, label="predicted angle +100 ms")
plt.axhline(-0.15, linestyle="--", linewidth=1)
plt.axhline(+0.18, linestyle="--", linewidth=1)
plt.xlabel("step"); plt.ylabel("radians")
plt.title("Predictive balance signal"); plt.legend(); plt.grid(True, alpha=0.25); plt.show()

plt.figure(figsize=(12, 4))
plt.plot(risk_history)
plt.xlabel("step"); plt.ylabel("risk 0..1"); plt.ylim(-0.02, 1.02)
plt.title("Tilt/speed indicator (not calibrated fall probability)"); plt.grid(True, alpha=0.25); plt.show()

print("MP4:", VIDEO_PATH)
print("GIF :", GIF_PATH)
print("log :", LOG_PATH)

## Validation and limitations

The optional cell below exercises the exact motor controller with fixed modes, rapidly cycling modes, and seeded random mode requests. It checks ten terrain seeds and reports distance, reward, and whether a run fell, finished, or reached its time limit. It does not contact Jev or measure API quality.

The speed parameters are deliberately narrow: large speed changes and a standing blend can disrupt the gait. These are heuristic settings for normal BipedalWalker, not a guarantee for new seeds or Hardcore terrain. Increase `MAX_STEPS` to extend the time allowance; falls still terminate immediately. The motor loop is 50 Hz in simulated time; synchronous API calls make wall-clock playback slower.

Verified locally with Gymnasium 1.3.0: all 60 offline runs (seeds 0–9; four fixed modes, cycling and seeded random requests) finished the course in 1,633–1,822 steps. Mean reward per schedule: 312.75–313.01. No live API validation was performed.


In [ ]:
#@title 6. Optional offline regression (no API requests)
def offline_regression(seeds=range(10)):
    results = []
    for schedule in [*MODE_SPEED, "cycle", "random"]:
        for seed in seeds:
            test_env = gym.make("BipedalWalker-v3", max_episode_steps=MAX_STEPS)
            try:
                obs, _ = test_env.reset(seed=seed)
                policy = new_reference()
                rng = np.random.default_rng(seed)
                mode = "cruise"
                reward_sum = 0.0
                start = float(test_env.unwrapped.hull.position.x)
                for step in range(MAX_STEPS):
                    if step % DECISION_EVERY == 0:
                        names = list(MODE_SPEED)
                        mode = (names[(step // DECISION_EVERY) % len(names)] if schedule == "cycle"
                                else str(rng.choice(names)) if schedule == "random" else schedule)
                    action, _, _, _ = gait_action(policy, obs, step, mode)
                    assert action.shape == (4,) and np.isfinite(action).all() and np.max(np.abs(action)) <= 1
                    obs, reward, terminated, truncated, _ = test_env.step(action)
                    reward_sum += float(reward)
                    if terminated or truncated:
                        break
                fallen = bool(test_env.unwrapped.game_over or test_env.unwrapped.hull.position.x < 0)
                result = dict(schedule=schedule, seed=int(seed), steps=step+1,
                              reward=round(reward_sum, 2), distance=round(float(test_env.unwrapped.hull.position.x)-start, 2),
                              outcome="fall" if fallen else "finish" if terminated else "time_limit")
                results.append(result)
            finally:
                test_env.close()
        group = [r for r in results if r["schedule"] == schedule]
        print(schedule, "| finished:", sum(r["outcome"] == "finish" for r in group),
              "/", len(group), "| minimum steps:", min(r["steps"] for r in group),
              "| mean reward:", round(float(np.mean([r["reward"] for r in group])), 2), flush=True)
    return results

offline_results = offline_regression()
